In [1]:
from pathlib import Path
from pprint import pprint
import asyncio
import graphrag.api as api
import pandas as pd
from graphrag.config.load_config import load_config
from graphrag.index.typing.pipeline_run_result import PipelineRunResult

14:33:55 - LiteLLM:WARNING: common_utils.py:979 - litellm: could not pre-load bedrock-runtime response stream shape — Bedrock event-stream decoding will be unavailable. Error: No module named 'botocore'
14:33:55 - LiteLLM:WARNING: common_utils.py:24 - litellm: could not pre-load sagemaker-runtime response stream shape — SageMaker event-stream decoding will be unavailable. Error: No module named 'botocore'


In [2]:
PROJECT_DIRECTORY = "."

graphrag_config = load_config(Path(PROJECT_DIRECTORY))

entities = pd.read_parquet(f"{PROJECT_DIRECTORY}/output/entities.parquet")
communities = pd.read_parquet(f"{PROJECT_DIRECTORY}/output/communities.parquet")
community_reports = pd.read_parquet(
    f"{PROJECT_DIRECTORY}/output/community_reports.parquet"
)

In [3]:
query = "A truck driver is injured during an accident.Which policy documents are relevant and why?"

In [34]:
response = ""
grcontext = ""

In [4]:
async def main():
    result, context = await api.global_search(
        config=graphrag_config,
        entities=entities,
        communities=communities,
        community_reports=community_reports,
        community_level=2,
        dynamic_community_selection=False,
        response_type="Multiple Paragraphs",
        query=query,
    )

    return result, context

In [5]:
res, con = asyncio.run(main())

In [6]:
con.keys()

dict_keys(['reports'])

In [7]:
con['reports']

,id,title,occurrence weight,content,rank
0,4,Insurance Regulatory Community in India,1.000000,# Insurance Regulatory Community in India\n\nT...,8.5
1,1,Insurance Regulatory Community in India,1.000000,# Insurance Regulatory Community in India\n\nT...,8.0
2,0,ICICI Lombard Insurance Community,1.000000,# ICICI Lombard Insurance Community\n\nThe ICI...,7.0
3,3,Mumbai's Insurance Community: ICICI Lombard an...,0.500000,# Mumbai's Insurance Community: ICICI Lombard ...,8.0
4,5,CIOINS and Insurance Ombudsman Community,0.333333,# CIOINS and Insurance Ombudsman Community\n\n...,7.5
5,2,Insurance Community Overview: ICICI Lombard,0.333333,# Insurance Community Overview: ICICI Lombard\...,7.5


In [8]:
selected_community_ids = con["reports"]["id"].tolist()

In [9]:
selected_community_ids = [int(x) for x in selected_community_ids]
selected_community_ids


[4, 1, 0, 3, 5, 2]

In [10]:
def get_nodes(df):
    print(df.columns)
    # nodes = df.dtypes
    nodes = df[
        df["community"].isin(selected_community_ids)
    ]
    return nodes

In [12]:
nodes = get_nodes(communities)
# nodes = get_nodes(community_reports)
nodes

Index(['id', 'human_readable_id', 'community', 'level', 'parent', 'children',
       'title', 'entity_ids', 'relationship_ids', 'text_unit_ids', 'period',
       'size'],
      dtype='object')


,id,human_readable_id,community,level,parent,children,title,entity_ids,relationship_ids,text_unit_ids,period,size
0,636a79b0-c2f5-42ae-b2a9-336f42d97392,0,0,0,-1,[],Community 0,"[aebcaa37-52b9-474f-a7d6-f837626b1c71, 3284682...","[016f78ad-b74c-4505-9307-4a3aadf80198, 26275dd...",[5045604fbd828f598b258fba3faeefc548810b88ed54e...,2026-06-08,8
1,07fbc6b2-a76f-430a-96b2-d3dd54bea139,1,1,0,-1,"[4, 5]",Community 1,"[2c97441d-5ea5-44c0-a964-f1f4357631e0, d29635b...","[08e2e3da-f2ae-4ab5-a8df-079798ebcf71, 0ae26e4...",[2b486bb2de58be8bd614dcc50eb6e106d55bd5b25ad28...,2026-06-08,10
2,9be50d9d-7742-490a-9bd4-3d03ba156422,2,2,0,-1,[],Community 2,"[ab0265ab-d91a-44d1-920a-85ecfa299880, b9e4710...","[69efe9ed-59ed-4861-9c6d-c9c3f022b0b8, e5d61e1...",[fa89b8a377cdd8cd24d9fc806a8d0f77892c7d29fc7be...,2026-06-08,6
3,1bb0c0ac-98e8-4f94-9dbd-6f0f7f5fe51f,3,3,0,-1,[],Community 3,"[4bd288ac-667c-4074-95bd-2987c440b8c6, 7251d6d...","[71e2e5d3-20f9-4194-b65d-e55ac1d22b64, 89f90f5...",[d81ec8d26068e6fe615233e8a1a3fedf1d150c5d3efbc...,2026-06-08,4
4,8d57e908-9ec5-4ff9-849a-271ef747bd9c,4,4,1,1,[],Community 4,"[2c97441d-5ea5-44c0-a964-f1f4357631e0, 8fd98c7...","[08e2e3da-f2ae-4ab5-a8df-079798ebcf71, 0ae26e4...",[2b486bb2de58be8bd614dcc50eb6e106d55bd5b25ad28...,2026-06-08,8
5,8d42448e-dd24-49b7-940a-ff1ac8482f68,5,5,1,1,[],Community 5,"[fe117966-d06f-4cf7-b3d5-faed8a4a8513, d29635b...",[bad9f210-b8fe-45a2-800d-6c8c69bb1ae3],[5d6cb8e18800c593cea559b2399feceb45ddd7e119aae...,2026-06-08,2
